# 09. Baseline ML còn thiếu trên NF-CSE-CIC-IDS2018-v3

Baseline ML trên đặc trưng node (task 4.1), val, 3 seed. Notebook 05 bị dừng ở giới hạn 4 giờ sau khi xong logreg, random forest (6,6 triệu node train nên rất chậm); notebook này chạy phần còn lại (XGBoost, LightGBM, MLP) ở chế độ nhanh: ít cấu hình thử hơn, MLP 25 vòng.

In [ ]:
import glob, os, subprocess, sys

def run(cmd, ok=(0,), **kw):
    r = subprocess.run(cmd, capture_output=True, text=True, **kw)
    print(r.stdout[-9000:], end="")
    print(r.stderr[-2500:], end="", file=sys.stderr)
    print(f"[mã thoát {r.returncode}]")
    if r.returncode not in ok:
        raise RuntimeError(f"Lệnh lỗi: {' '.join(cmd)}")

REPO = "/tmp/gnn-iot-ids"
run(["rm", "-rf", REPO])
run(["git", "clone", "--depth", "1", "https://github.com/minucedev/gnn-iot-ids.git", REPO])
run(["git", "-C", REPO, "log", "-1", "--format=%H %s"])
with open(f"{REPO}/requirements.txt") as f:
    reqs = [l.strip() for l in f if l.strip() and not l.startswith("#")]
keep = [r for r in reqs if r.split("==")[0].lower() not in {"torch", "jupyter", "ipykernel"}]
run([sys.executable, "-m", "pip", "install", "-q", *keep])
print(os.cpu_count(), "CPU")

In [ ]:
OUT, ds = "/kaggle/working", "nf_cicids2018_v3"
d = os.path.dirname(next(glob.iglob(f"/kaggle/input/**/{ds}/split.json", recursive=True)))
print("=====", ds, d)
run([sys.executable, "-I", "scripts/run_baselines.py", d, ds, f"{OUT}/results_baselines_{ds}.csv",
     f"{OUT}/baseline_search_{ds}.json", "models=xgboost,lightgbm,mlp", "fast=1"], cwd=REPO)